# 👗 Garment Image Beautifier (nano banana + studio compositing)

Two stages for **consistent** results:
1. **nano banana** (Gemini 2.5 Flash Image) cleans each garment — flat lay, de-wrinkled, logos preserved — on plain white.
2. **Code** cuts out the garment and places it dead-center on an identical `#D7D7D7` 1:1 square with one uniform soft shadow.

Reads from the `input/` folder of your GitHub repo, saves results to `output/`. No Google Drive.

Run each cell in order (▶️ on the left, top to bottom). You'll paste your Gemini API key and a GitHub token.

## Step 1 — Install the tools
Run once. ~1 minute (the background-remover is a bit bigger).

In [ ]:
!pip install -q google-genai pillow requests rembg onnxruntime
print('✅ Done installing.')

## Step 2 — Paste your Gemini API key
Get one at https://aistudio.google.com/apikey. A box appears when you run this — paste your key there (it stays hidden).

In [ ]:
from getpass import getpass
API_KEY = getpass('Paste your Gemini API key and press Enter: ')
print('✅ Gemini key saved for this session.')

## Step 3 — Paste your GitHub token
Lets the notebook save results into your repo's `output/` folder.

**To make one:** GitHub → Settings → Developer settings → **Fine-grained personal access tokens** → Generate new token →
- Repository access: **Only select repositories** → pick `garment-images`
- Permissions → Repository permissions → **Contents: Read and write**
- Generate, copy the token (starts with `github_pat_`), paste it in the box.

In [ ]:
GITHUB_TOKEN = getpass('Paste your GitHub token and press Enter: ')
print('✅ GitHub token saved for this session.')

## Step 4 — Settings
The AI prompt (stage 1) and the studio look (stage 2) are both set here. Tweak the numbers to taste.

In [ ]:
# --- STAGE 1: what the AI does (clean the garment on plain white) ---
PROMPT = """
A clean, professional flat-lay product photo of the exact garment shown in the provided image, viewed from directly above.
Lay the garment completely flat and perfectly symmetrical, fully de-wrinkled and smoothed, spread out so the entire item is visible and uncropped.
Place it on a seamless, pure solid white background with soft, even, diffused studio lighting and sharp focus across the whole garment, capturing true fabric texture.
Preserve every authentic brand detail, logo, tag, printed text and symbol exactly as it appears — do NOT add, remove, invent, or alter any logo, text, or symbol.
No drop shadow, no props, no mannequin, no hanger — just the garment on plain white.
"""

# --- STAGE 2: the locked-down studio look (identical for every image) ---
CANVAS_SIZE   = 2048              # final square size in pixels (1:1)
BG_HEX        = '#D7D7D7'         # exact background color
MARGIN        = 0.12             # empty space around the garment (0.12 = 12% each side)
SHADOW_BLUR   = 45               # softness of the shadow (bigger = softer)
SHADOW_OPACITY = 60             # shadow darkness, 0 (invisible) to 255 (black)
SHADOW_GROW   = 8                # how far the shadow spreads past the garment edge (px)

# --- Your GitHub repo ---
GITHUB_REPO = 'michellefleek/garment-images'
GITHUB_BRANCH = 'main'
INPUT_PATH = 'input'
OUTPUT_PATH = 'output'

MODEL = 'gemini-2.5-flash-image'

# 🧪 TEST MODE: only process this many images. Set to None for ALL.
TEST_LIMIT = 5

print('✅ Settings loaded.')

## Step 5 — Run the batch
Cleans each garment with the AI, composites it onto the identical studio square, and saves to `output/`.

Safe to stop and re-run — images already in `output/` are skipped.

In [ ]:
import time, io, base64, json, requests
from PIL import Image, ImageFilter
from google import genai
from rembg import remove, new_session

client = genai.Client(api_key=API_KEY)
GH = {'Authorization': f'Bearer {GITHUB_TOKEN}', 'Accept': 'application/vnd.github+json'}
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
MAX_RETRIES = 4
BG_RGB = tuple(int(BG_HEX.lstrip('#')[i:i+2], 16) for i in (0, 2, 4))
rembg_session = new_session('u2net')

def list_files(path):
    url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{path}?ref={GITHUB_BRANCH}'
    r = requests.get(url, headers=GH)
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return [it for it in r.json() if it['type'] == 'file']

def ai_clean(img_bytes):
    img = Image.open(io.BytesIO(img_bytes))
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.models.generate_content(model=MODEL, contents=[PROMPT, img])
            for part in resp.candidates[0].content.parts:
                if getattr(part, 'inline_data', None) and part.inline_data.data:
                    return part.inline_data.data
            return None
        except Exception as e:
            wait = min(2 ** attempt, 30)
            print(f'   ⚠️  AI attempt {attempt} failed ({e}); retrying in {wait}s...')
            time.sleep(wait)
    return None

def studio_composite(clean_png):
    # 1. cut the garment out of the AI's white background
    cut = remove(clean_png, session=rembg_session)
    fg = Image.open(io.BytesIO(cut)).convert('RGBA')
    fg = fg.crop(fg.getbbox())  # trim to the garment
    # 2. scale to fit inside the margins
    max_dim = int(CANVAS_SIZE * (1 - 2 * MARGIN))
    fg.thumbnail((max_dim, max_dim), Image.LANCZOS)
    x = (CANVAS_SIZE - fg.width) // 2
    y = (CANVAS_SIZE - fg.height) // 2
    # 3. solid background canvas
    canvas = Image.new('RGBA', (CANVAS_SIZE, CANVAS_SIZE), BG_RGB + (255,))
    # 4. uniform soft shadow, centered (x=0, y=0 offset)
    mask = Image.new('L', (CANVAS_SIZE, CANVAS_SIZE), 0)
    mask.paste(fg.split()[3], (x, y))
    if SHADOW_GROW:
        mask = mask.filter(ImageFilter.MaxFilter(SHADOW_GROW * 2 + 1))
    mask = mask.filter(ImageFilter.GaussianBlur(SHADOW_BLUR))
    mask = mask.point(lambda p: int(p * SHADOW_OPACITY / 255))
    shadow = Image.new('RGBA', (CANVAS_SIZE, CANVAS_SIZE), (30, 30, 30, 0))
    shadow.putalpha(mask)
    canvas = Image.alpha_composite(canvas, shadow)
    # 5. garment on top
    canvas.paste(fg, (x, y), fg)
    out = io.BytesIO()
    canvas.convert('RGB').save(out, 'PNG')
    return out.getvalue()

def save_to_github(name, data_bytes):
    api = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{OUTPUT_PATH}/{name}'
    existing = requests.get(f'{api}?ref={GITHUB_BRANCH}', headers=GH)
    sha = existing.json().get('sha') if existing.status_code == 200 else None
    payload = {'message': f'Add beautified {name}', 'branch': GITHUB_BRANCH,
               'content': base64.b64encode(data_bytes).decode()}
    if sha:
        payload['sha'] = sha
    r = requests.put(api, headers=GH, data=json.dumps(payload))
    r.raise_for_status()

inputs = [it for it in list_files(INPUT_PATH) if it['name'].lower().endswith(IMAGE_EXTS)]
already_done = {it['name'] for it in list_files(OUTPUT_PATH)}
if TEST_LIMIT:
    inputs = inputs[:TEST_LIMIT]
    print(f'🧪 TEST MODE: only processing the first {TEST_LIMIT} image(s).')
print(f'Found {len(inputs)} image(s) to process.\n')

done = skipped = failed = 0
for i, it in enumerate(inputs, 1):
    name = it['name']
    out_name = name.rsplit('.', 1)[0] + '.png'
    if out_name in already_done:
        skipped += 1
        print(f'[{i}/{len(inputs)}] ⏭️  {name} (already in output/)')
        continue
    print(f'[{i}/{len(inputs)}] 🎨 {name} ...')
    img_bytes = requests.get(it['download_url']).content
    clean = ai_clean(img_bytes)
    if not clean:
        failed += 1
        print('          ❌ AI could not clean this one (skipping)')
        continue
    try:
        final = studio_composite(clean)
        save_to_github(out_name, final)
        done += 1
        print(f'          ✅ saved to output/{out_name}')
    except Exception as e:
        failed += 1
        print(f'          ❌ compositing failed ({e})')
    time.sleep(1)

print(f'\n🏁 Finished. {done} new, {skipped} already done, {failed} failed.')
print(f'Results: https://github.com/{GITHUB_REPO}/tree/{GITHUB_BRANCH}/{OUTPUT_PATH}')